# Few-Shot Classification
Prerequisites: labeled examples and basic Python. Estimated time: 30 minutes. Objectives: construct a small labeled demonstration set, classify by a transparent similarity baseline, and evaluate coverage and failure cases.

## Concept recap and assumptions
Few-shot learning uses a small labeled support set to infer a task for new examples. Here, token overlap is a deliberately simple, deterministic stand-in for a learned model; it teaches support-set handling, not language understanding or model performance.

In [ ]:
def words(text):
    return set(text.lower().split())

def classify(query, demonstrations):
    if not demonstrations:
        raise ValueError('at least one labeled demonstration is required')
    q = words(query)
    ranked = []
    for index, (example, label) in enumerate(demonstrations):
        if not label:
            raise ValueError('labels must be non-empty')
        overlap = len(q & words(example))
        ranked.append((overlap, -index, label))
    return max(ranked)[2]

train = [('great helpful service', 'positive'), ('broken slow delivery', 'negative'), ('helpful fast support', 'positive'), ('late damaged parcel', 'negative')]
held_out = [('helpful service', 'positive'), ('slow damaged parcel', 'negative'), ('great support', 'positive')]
predictions = [classify(text, train) for text, _ in held_out]
predictions

## Baseline assertions
The baseline picks the label on the demonstration with greatest word overlap. Stable input order breaks ties. This makes the decision inspectable and exposes how a tiny or unbalanced support set can dominate results.

In [ ]:
accuracy = sum(pred == gold for pred, (_, gold) in zip(predictions, held_out)) / len(held_out)
assert predictions == ['positive', 'negative', 'positive']
assert accuracy == 1.0
assert classify('slow', train) == 'negative'

## Comparison and metric
Accuracy on this tiny held-out set is a descriptive toy metric, not a reliable estimate. In practice, keep support examples separate from evaluation data, compare zero-shot and few-shot prompts on the same cases, and report per-class scores because overall accuracy hides class imbalance.

In [ ]:
# Edge/failure cases: no support examples and a query with no lexical match.
try:
    classify('unknown vocabulary', [])
except ValueError as error:
    assert 'demonstration' in str(error)
else:
    raise AssertionError('empty support set must fail clearly')
assert classify('unmatched words', train) == 'positive'  # stable first-example tie fallback, not justified confidence
# A duplicated label can bias the support set; count labels before deployment.
label_counts = {label: sum(y == label for _, y in train) for label in {y for _, y in train}}
assert label_counts == {'positive': 2, 'negative': 2}

## Exercises
1. Remove one negative demonstration and compare per-class accuracy. 2. Replace overlap with Jaccard similarity and specify the zero-union behavior. 3. Add a confidence abstention rule for zero overlap. Expected properties: a fixed support set gives repeatable predictions, empty support is rejected, and evaluation examples are not reused as demonstrations.

## Summary, next lesson, and practice
Few-shot work depends on selecting representative labeled examples and testing generalization beyond them. This lexical baseline is not an LLM and cannot establish semantic performance. Next: In-Context Learning, which emphasizes how examples are assembled in the prompt. Practice: [Few-shot interview drill](../interview-prep/curriculum-practice.md#few-shot-learning).